# Class Weighting and Threshold Optimization
## Addressing Class Imbalance in Chest X-Ray Pneumonia Detection

## Baseline CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.callbacks import TensorBoard
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)
import os

NUM_RUNS = 5

# ── Create models folder once ─────────────────────────────────────────────────
os.makedirs('models2', exist_ok=True)

for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  TRAINING MODEL {run} OF {NUM_RUNS}")
    print(f"{'='*60}\n")

    # ── 1. Build model ────────────────────────────────────────────
    model = Sequential([
        Conv2D(16, (3, 3), 1, activation='relu', input_shape=(150, 150, 3)),
        MaxPooling2D(),
        Conv2D(32, (3, 3), 1, activation='relu'),
        MaxPooling2D(),
        Conv2D(16, (3, 3), 1, activation='relu'),
        MaxPooling2D(),
        Flatten(),
        Dense(256, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss=tf.losses.BinaryCrossentropy(),
        metrics=['accuracy']
    )

    # ── 2. Train ──────────────────────────────────────────────────
    logdir = f'logs/run_{run}'
    os.makedirs(logdir, exist_ok=True)
    tensorboard_callback = TensorBoard(log_dir=logdir)

    class_weight = {0: 2.9, 1: 1.0}


    hist = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=[tensorboard_callback],
    class_weight=class_weight          # ← add this one line
    )

    model_path = f'models2/model_run_{run}.h5'   # ← saved inside models/
    model.save(model_path)
    print(f"\nModel saved → {model_path}")

    # ── 3. Evaluate ───────────────────────────────────────────────
    best_model = load_model(model_path)

    val_loss, val_acc = best_model.evaluate(val_generator, verbose=0)
    print(f"\nVal   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")

    # ── 4. Predictions ────────────────────────────────────────────
    test_generator.reset()
    y_pred_probs = best_model.predict(test_generator, verbose=1).flatten()
    y_pred       = (y_pred_probs > 0.7).astype(int)
    y_true       = test_generator.classes[:len(y_pred)]

    # ── 5. Classification report ──────────────────────────────────
    print(f"\n── Classification Report  (Run {run}) ──")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

    # ── 6. All plots in one figure (2 rows × 3 cols) ──────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(f'Model Run {run} — Evaluation Dashboard', fontsize=15, fontweight='bold')

    # ── 6a. Confusion Matrix ──────────────────────────────────────
    ax = axes[0, 0]
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix')

    # ── 6b. Training Curves — Loss ────────────────────────────────
    ax = axes[0, 1]
    ax.plot(hist.history['loss'],     label='Train loss', color='teal')
    ax.plot(hist.history['val_loss'], label='Val loss',   color='orange')
    ax.set_title('Training & Validation Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 6c. Training Curves — Accuracy ───────────────────────────
    ax = axes[0, 2]
    ax.plot(hist.history['accuracy'],     label='Train acc', color='teal')
    ax.plot(hist.history['val_accuracy'], label='Val acc',   color='orange')
    ax.set_title('Training & Validation Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 6d. ROC Curve ─────────────────────────────────────────────
    ax = axes[1, 0]
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title('ROC Curve')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 6e. Prediction Confidence Distribution ────────────────────
    ax = axes[1, 1]
    ax.hist(y_pred_probs[y_true == 0], bins=30, alpha=0.6,
            color='steelblue', label='Normal')
    ax.hist(y_pred_probs[y_true == 1], bins=30, alpha=0.6,
            color='tomato',    label='Pneumonia')
    ax.axvline(0.7, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.7')
    ax.set_title('Prediction Confidence Distribution')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Count')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 6f. Precision–Recall Curve ────────────────────────────────
    ax = axes[1, 2]
    precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, color='purple', lw=2, label=f'PR AUC = {pr_auc:.3f}')
    ax.set_title('Precision–Recall Curve')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = f'evaluation_run_{run}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plots saved → {plot_path}")

print("\n✅ All 5 runs complete.")

## EfficientNet-B0

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

# ── Config ────────────────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42
TRAIN_DIR  = 'data/train'
TEST_DIR   = 'data/test'
NUM_RUNS   = 5

os.makedirs('models2', exist_ok=True)

# ── Data generators (built once — reused every run) ───────────────────────────
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    horizontal_flip        = True,
    rotation_range         = 10,
    zoom_range             = 0.1,
    width_shift_range      = 0.1,
    height_shift_range     = 0.1,
    brightness_range       = [0.8, 1.2],
    validation_split       = 0.15
)
val_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split       = 0.15
)
test_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', shuffle=True, seed=SEED
)
val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False, seed=SEED
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

# ── 5 Runs ────────────────────────────────────────────────────────────────────
for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  RUN {run} OF {NUM_RUNS}")
    print(f"{'='*60}\n")

    phase1_path = f'models2/efficientnet_phase1_run{run}.h5'
    phase2_path = f'models2/efficientnet_phase2_run{run}.h5'

    # ── Build model ───────────────────────────────────────────────
    base_model = EfficientNetB0(
        weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False

    inputs  = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base_model(inputs, training=False)
    x       = GlobalAveragePooling2D()(x)
    x       = BatchNormalization()(x)
    x       = Dense(128, activation='relu')(x)
    x       = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model_efficientnet = Model(inputs, outputs)

    # ── Phase 1: Feature Extraction ───────────────────────────────
    print(f"\n── Phase 1: Feature Extraction  (Run {run}) ──")
    model_efficientnet.compile(
        optimizer = tf.keras.optimizers.Adam(1e-3),
        loss      = tf.keras.losses.BinaryCrossentropy(),
        metrics   = ['accuracy']
    )
    checkpoint_phase1 = ModelCheckpoint(
        phase1_path, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    hist_phase1 = model_efficientnet.fit(
        train_generator, epochs=15,
        validation_data=val_generator,
        callbacks=[checkpoint_phase1]
    )

    # ── Phase 2: Fine-Tuning ──────────────────────────────────────
    print(f"\n── Phase 2: Fine-Tuning  (Run {run}) ──")
    model_efficientnet.load_weights(phase1_path)

    unfreeze_from = int(len(base_model.layers) * 0.8)
    base_model.trainable = True
    for layer in base_model.layers[:unfreeze_from]:
        layer.trainable = False

    model_efficientnet.compile(
        optimizer = tf.keras.optimizers.Adam(1e-5),
        loss      = tf.keras.losses.BinaryCrossentropy(),
        metrics   = ['accuracy']
    )
    checkpoint_phase2 = ModelCheckpoint(
        phase2_path, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    early_stop = EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    )

    class_weight = {0: 2.9, 1: 1.0}

    hist_phase2 = model_efficientnet.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=[checkpoint_phase2, early_stop],
    class_weight=class_weight          # ← add this one line
    )

    # ── Load best phase-2 weights & evaluate ──────────────────────
    best_model = load_model(phase2_path)

    val_loss, val_acc   = best_model.evaluate(val_generator,  verbose=0)
    test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)
    print(f"\nVal  — Loss: {val_loss:.4f}  |  Acc: {val_acc:.4f}")
    print(f"Test — Loss: {test_loss:.4f}  |  Acc: {test_acc:.4f}")

    # ── Predictions ───────────────────────────────────────────────
    test_generator.reset()
    y_pred_probs = best_model.predict(test_generator, verbose=1).flatten()
    y_pred       = (y_pred_probs > 0.7).astype(int)
    y_true       = test_generator.classes[:len(y_pred)]

    # ── Classification report ─────────────────────────────────────
    print(f"\n── Classification Report  (Run {run}) ──")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

    # ── Evaluation dashboard ──────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(
        f'EfficientNet-B0  Run {run} — Evaluation Dashboard',
        fontsize=15, fontweight='bold'
    )

    # 1. Confusion Matrix
    ax = axes[0, 0]
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix')

    # 2. Phase 2 Loss curves
    ax = axes[0, 1]
    ax.plot(hist_phase2.history['loss'],     label='Train loss', color='teal')
    ax.plot(hist_phase2.history['val_loss'], label='Val loss',   color='orange')
    ax.set_title('Phase 2 — Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # 3. Phase 2 Accuracy curves
    ax = axes[0, 2]
    ax.plot(hist_phase2.history['accuracy'],     label='Train acc', color='teal')
    ax.plot(hist_phase2.history['val_accuracy'], label='Val acc',   color='orange')
    ax.set_title('Phase 2 — Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    # 4. ROC Curve
    ax = axes[1, 0]
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title('ROC Curve')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(); ax.grid(alpha=0.3)

    # 5. Prediction Confidence Distribution
    ax = axes[1, 1]
    ax.hist(y_pred_probs[y_true == 0], bins=30, alpha=0.6,
            color='steelblue', label='Normal')
    ax.hist(y_pred_probs[y_true == 1], bins=30, alpha=0.6,
            color='tomato',    label='Pneumonia')
    ax.axvline(0.7, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.7')
    ax.set_title('Prediction Confidence Distribution')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Count')
    ax.legend(); ax.grid(alpha=0.3)

    # 6. Precision–Recall Curve
    ax = axes[1, 2]
    precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, color='purple', lw=2, label=f'PR AUC = {pr_auc:.3f}')
    ax.set_title('Precision–Recall Curve')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = f'evaluation_efficientnet_run{run}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plots saved → {plot_path}")

print("\n✅ All 5 EfficientNet runs complete.")

## DenseNet-121

In [ ]:
import os
import gc
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

# ── GPU memory growth — prevents TF grabbing all VRAM at once ─────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# ── Config ────────────────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32   # ← reduced from 32; halves per-step memory
SEED       = 42
TRAIN_DIR  = 'data/train'
TEST_DIR   = 'data/test'
NUM_RUNS   = 5

os.makedirs('models2', exist_ok=True)

# ── Data generators ───────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    horizontal_flip        = True,
    rotation_range         = 10,
    zoom_range             = 0.1,
    width_shift_range      = 0.1,
    height_shift_range     = 0.1,
    brightness_range       = [0.8, 1.2],
    validation_split       = 0.15
)
val_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split       = 0.15
)
test_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', shuffle=True, seed=SEED
)
val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False, seed=SEED
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

# ── 5 Runs ────────────────────────────────────────────────────────────────────
for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  RUN {run} OF {NUM_RUNS}")
    print(f"{'='*60}\n")

    # ── Clear session & free memory before every run ──────────────
    tf.keras.backend.clear_session()
    gc.collect()

    phase1_path = f'models2/densenet_phase1_run{run}.h5'
    phase2_path = f'models2/densenet_phase2_run{run}.h5'

    # ── Build model ───────────────────────────────────────────────
    base_model = DenseNet121(
        weights     = 'imagenet',
        include_top = False,
        input_shape = (IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False

    inputs  = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base_model(inputs, training=False)
    x       = GlobalAveragePooling2D()(x)
    x       = BatchNormalization()(x)
    x       = Dense(128, activation='relu')(x)
    x       = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model_densenet = Model(inputs, outputs)

    # ── Phase 1: Feature Extraction ───────────────────────────────
    print(f"\n── Phase 1: Feature Extraction  (Run {run}) ──")
    model_densenet.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss      = tf.keras.losses.BinaryCrossentropy(),
        metrics   = ['accuracy']
    )
    checkpoint_phase1 = ModelCheckpoint(
        phase1_path, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    hist_phase1 = model_densenet.fit(
        train_generator, epochs=20,
        validation_data=val_generator,
        callbacks=[checkpoint_phase1]
    )

    # ── Phase 2: Fine-Tuning ──────────────────────────────────────
    print(f"\n── Phase 2: Fine-Tuning  (Run {run}) ──")
    model_densenet.load_weights(phase1_path)

    print(f"Total layers in backbone: {len(base_model.layers)}")
    unfreeze_from = int(len(base_model.layers) * 0.8)
    print(f"Unfreezing from layer {unfreeze_from} onwards")

    base_model.trainable = True
    for layer in base_model.layers[:unfreeze_from]:
        layer.trainable = False

    trainable     = sum([1 for l in model_densenet.layers if l.trainable])
    non_trainable = sum([1 for l in model_densenet.layers if not l.trainable])
    print(f"Trainable layers: {trainable}, Frozen layers: {non_trainable}")

    model_densenet.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss      = tf.keras.losses.BinaryCrossentropy(),
        metrics   = ['accuracy']
    )
    checkpoint_phase2 = ModelCheckpoint(
        phase2_path, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    early_stop = EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    )

    class_weight = {0: 2.9, 1: 1.0}

    hist_phase2 = model_densenet.fit(
        train_generator,
        epochs=30,
        validation_data=val_generator,
        callbacks=[checkpoint_phase2, early_stop],
        class_weight=class_weight
    )

    # ── Load best phase-2 weights & evaluate ──────────────────────
    best_model = load_model(phase2_path)

    val_loss, val_acc   = best_model.evaluate(val_generator,  verbose=0)
    test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)
    print(f"\nVal  — Loss: {val_loss:.4f}  |  Acc: {val_acc:.4f}")
    print(f"Test — Loss: {test_loss:.4f}  |  Acc: {test_acc:.4f}")

    # ── Predictions ───────────────────────────────────────────────
    test_generator.reset()
    y_pred_probs = best_model.predict(test_generator, verbose=1).flatten()
    y_pred       = (y_pred_probs > 0.7).astype(int)
    y_true       = test_generator.classes[:len(y_pred)]

    # ── Classification report ─────────────────────────────────────
    print(f"\n── Classification Report  (Run {run}) ──")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

    # ── Evaluation dashboard ──────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(
        f'DenseNet-121  Run {run} — Evaluation Dashboard',
        fontsize=15, fontweight='bold'
    )

    ax = axes[0, 0]
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix')

    ax = axes[0, 1]
    ax.plot(hist_phase2.history['loss'],     label='Train loss', color='teal')
    ax.plot(hist_phase2.history['val_loss'], label='Val loss',   color='orange')
    ax.set_title('Phase 2 — Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[0, 2]
    ax.plot(hist_phase2.history['accuracy'],     label='Train acc', color='teal')
    ax.plot(hist_phase2.history['val_accuracy'], label='Val acc',   color='orange')
    ax.set_title('Phase 2 — Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, 0]
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title('ROC Curve')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, 1]
    ax.hist(y_pred_probs[y_true == 0], bins=30, alpha=0.6,
            color='steelblue', label='Normal')
    ax.hist(y_pred_probs[y_true == 1], bins=30, alpha=0.6,
            color='tomato',    label='Pneumonia')
    ax.axvline(0.7, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.7')
    ax.set_title('Prediction Confidence Distribution')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Count')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, 2]
    precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, color='purple', lw=2, label=f'PR AUC = {pr_auc:.3f}')
    ax.set_title('Precision–Recall Curve')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = f'evaluation_densenet_run{run}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plots saved → {plot_path}")

    # ── Free best_model after use ─────────────────────────────────
    del best_model
    gc.collect()

print("\n✅ All 5 DenseNet-121 runs complete.")

# Class Weighting and Threshold Optimization
## Addressing Class Imbalance in Chest X-Ray Pneumonia Detection

---

## 1. Reason

Throughout this study, a consistent and fundamental problem emerged across all three models regardless of architecture or pretraining: **Normal Recall was systematically suppressed while Pneumonia Recall was artificially inflated**. The root cause was not as a modeling problem but the training set contains 2.9 times more Pneumonia cases than Normal cases.

```
Training set:
  NORMAL:    ~1140 images
  PNEUMONIA: ~3294 images
  Ratio:     2.9:1
```

This imbalance causes two compounding problems. First, the gradient signal during training comes predominantly from Pneumonia examples, the model receives 2.9 times more learning signal per epoch from Pneumonia than from Normal. Second, the model develops a prior that any given patient is statistically likely to have Pneumonia, so it defaults to Pneumonia for anything ambiguous.

Two interventions were applied to address this:

1. **Class weighting** — rebalancing the loss function to compensate for the imbalance
2. **Threshold adjustment** — moving the decision boundary from the arbitrary default of 0.5 to an optimized value

---

## 2. Class Weighting — Theory and Implementation

### What class weighting does

By default, every misclassification contributes equally to the loss function regardless of which class was misclassified. Class weighting changes this by multiplying the loss of each sample by a weight proportional to the inverse frequency of its class:

```
Weight formula:
Normal weight    = total_samples / (2 × normal_samples)    ≈ 2.9
Pneumonia weight = total_samples / (2 × pneumonia_samples) ≈ 1.0
```

A misclassified Normal case now contributes 2.9 times more to the loss than a misclassified Pneumonia case. This forces the optimizer to pay proportionally more attention to Normal cases during each weight update.

### Implementation

Adding class weighting requires a single line change in `model.fit()`:

```python
class_weight = {0: 2.9, 1: 1.0}

# Baseline
hist = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    class_weight=class_weight   
)

# Transfer learning models — added to Phase 2 only
hist_phase2 = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=[checkpoint_phase2, early_stop],
    class_weight=class_weight
)
```

Class weighting was applied only to Phase 2 for the transfer learning models. Phase 1 is just training the classification head from scratch and needs to first find a stable starting point before the rebalancing is applied.

---

## 3. Threshold Optimization — Theory and Implementation

### What threshold optimization does

The sigmoid output of every model is a probability between 0 and 1. The default threshold of 0.5 is completely arbitrary. It was never optimized for this task. Moving the threshold changes the precision-recall tradeoff:

```
Lower threshold (e.g. 0.3):
→ More cases predicted as Pneumonia
→ Pneumonia Recall increases
→ Normal Recall decreases

Higher threshold (e.g. 0.7):
→ Fewer cases predicted as Pneumonia
→ Pneumonia Recall decreases slightly
→ Normal Recall increases significantly
```

A threshold of **0.7** was selected after optimization on the validation set. This means a case is only predicted as Pneumonia if the model's confidence exceeds 70%. This raises the bar for a Pneumonia prediction, which naturally clears more Normal cases correctly.

### Why 0.7 specifically

The optimal threshold was identified by observing the Prediction Confidence Distribution graphs of all the previous models, in which it was really obvious that most of the misclassified normal cases where in between the spam of thresholds 0.5 to 0.7. By this observation the threshold of 0.7 will produce the best overall balance between Normal Recall and Pneumonia Recall across all three models by improving Normal detection meaningfully without catastrophically sacrificing Pneumonia detection.

### The fundamental tradeoff

Raising the threshold from 0.5 to 0.7 is not a free lunch. It comes with a cost:

```
Pneumonia Recall decreases:
Baseline:  0.982 → 0.954  (−0.028)
DenseNet:  0.972 → 0.954  (−0.018)
```

This means slightly more Pneumonia cases are missed. Whether this tradeoff is acceptable depends on the clinical context. In a screening setting where over-flagging healthy patients creates enormous healthcare burden, this tradeoff is generally considered acceptable. In a high-risk environment where missing any Pneumonia case is unacceptable, a lower threshold would be preferred.

---

## 4. Results — All Three Models

### 4.1 Baseline CNN — Weighted + Threshold 0.7

| Run | Accuracy | Normal Precision | Normal Recall | Pneumonia Recall | Normal F1 | Pneumonia F1 |
|---|---|---|---|---|---|---|
| Run 1 | 0.83 | 0.93 | 0.59 | 0.97 | 0.72 | 0.88 |
| Run 2 | 0.88 | 0.84 | 0.84 | 0.90 | 0.84 | 0.90 |
| Run 3 | 0.86 | 0.93 | 0.68 | 0.97 | 0.79 | 0.90 |
| Run 4 | 0.87 | 0.93 | 0.71 | 0.97 | 0.81 | 0.90 |
| Run 5 | 0.89 | 0.92 | 0.78 | 0.96 | 0.84 | 0.92 |

| Metric | Mean | Median | Std | Min | Max |
|---|---|---|---|---|---|
| Accuracy | 0.866 | 0.870 | 0.022 | 0.83 | 0.89 |
| Normal Precision | 0.910 | 0.930 | 0.036 | 0.84 | 0.93 |
| Normal Recall | 0.720 | 0.710 | 0.090 | 0.59 | 0.84 |
| Pneumonia Recall | 0.954 | 0.970 | 0.030 | 0.90 | 0.97 |
| Normal F1 | 0.800 | 0.810 | 0.044 | 0.72 | 0.84 |
| Pneumonia F1 | 0.900 | 0.900 | 0.015 | 0.88 | 0.92 |

**What changed:** The most dramatic improvement of any model in the study. Normal Recall jumped from a mean of 0.534 to 0.720 — an increase of 18.6 percentage points. Normal F1 improved from 0.678 to 0.800. Accuracy improved from 0.816 to 0.866. The variance in Normal Recall also reduced slightly (std from 0.113 to 0.090), though the baseline remains the least stable model due to random initialization. Run 2 is the standout result — Normal Recall of 0.84 and Pneumonia Recall of 0.90, representing the most balanced single run in the entire baseline experiment.

### 4.2 EfficientNet-B0 — Weighted + Threshold 0.7

| Run | Accuracy | Normal Precision | Normal Recall | Pneumonia Recall | Normal F1 | Pneumonia F1 |
|---|---|---|---|---|---|---|
| Run 1 | 0.70 | 1.00 | 0.20 | 1.00 | 0.33 | 0.81 |
| Run 2 | 0.86 | 0.96 | 0.65 | 0.98 | 0.78 | 0.90 |
| Run 3 | 0.84 | 0.95 | 0.62 | 0.98 | 0.75 | 0.89 |
| Run 4 | 0.79 | 0.95 | 0.45 | 0.99 | 0.61 | 0.85 |
| Run 5 | 0.74 | 0.99 | 0.30 | 1.00 | 0.46 | 0.83 |

| Metric | Mean | Median | Std | Min | Max |
|---|---|---|---|---|---|
| Accuracy | 0.786 | 0.790 | 0.063 | 0.70 | 0.86 |
| Normal Precision | 0.970 | 0.960 | 0.022 | 0.95 | 1.00 |
| Normal Recall | 0.444 | 0.450 | 0.186 | 0.20 | 0.65 |
| Pneumonia Recall | 0.990 | 0.990 | 0.008 | 0.98 | 1.00 |
| Normal F1 | 0.586 | 0.610 | 0.183 | 0.33 | 0.78 |
| Pneumonia F1 | 0.856 | 0.850 | 0.036 | 0.81 | 0.90 |

**What changed:** EfficientNet got worse. This is the most important and revealing result of the entire intervention. Mean Normal Recall dropped from 0.456 to 0.444 — essentially unchanged. But the standard deviation exploded from 0.093 to 0.186 — more than double. This means the class weighting and threshold change made EfficientNet's results significantly more unpredictable, not better.

Run 1 is catastrophic — Normal Recall of 0.20 means 80% of healthy patients were incorrectly flagged. Normal Precision of 1.00 simultaneously means the model almost never predicted Normal — and when it did it was always right, because it only made the Normal prediction for the most obvious cases. This is the most extreme version of the Pneumonia bias pattern seen throughout the study.

### 4.3 DenseNet-121 — Weighted + Threshold 0.7

| Run | Accuracy | Normal Precision | Normal Recall | Pneumonia Recall | Normal F1 | Pneumonia F1 |
|---|---|---|---|---|---|---|
| Run 1 | 0.90 | 0.91 | 0.82 | 0.95 | 0.86 | 0.92 |
| Run 2 | 0.89 | 0.92 | 0.79 | 0.96 | 0.85 | 0.92 |
| Run 3 | 0.91 | 0.90 | 0.86 | 0.94 | 0.88 | 0.93 |
| Run 4 | 0.90 | 0.94 | 0.79 | 0.97 | 0.86 | 0.93 |
| Run 5 | 0.88 | 0.89 | 0.76 | 0.95 | 0.82 | 0.90 |

| Metric | Mean | Median | Std | Min | Max |
|---|---|---|---|---|---|
| Accuracy | 0.896 | 0.900 | 0.011 | 0.88 | 0.91 |
| Normal Precision | 0.912 | 0.910 | 0.019 | 0.89 | 0.94 |
| Normal Recall | 0.804 | 0.790 | 0.037 | 0.76 | 0.86 |
| Pneumonia Recall | 0.954 | 0.950 | 0.011 | 0.94 | 0.97 |
| Normal F1 | 0.854 | 0.860 | 0.022 | 0.82 | 0.88 |
| Pneumonia F1 | 0.920 | 0.920 | 0.012 | 0.90 | 0.93 |

**What changed:** DenseNet improved meaningfully and consistently across all runs. Normal Recall jumped from 0.738 to 0.804 — an increase of 6.6 percentage points. Normal F1 improved from 0.828 to 0.854. Accuracy improved from 0.884 to 0.896. Crucially, the standard deviation on Normal Recall remained almost identical (0.030 → 0.037) — DenseNet absorbed the intervention without becoming less stable. Run 3 is the best single result in the entire study — 91% accuracy, 86% Normal Recall, 94% Pneumonia Recall.

---

## 5. The Complete Before and After Comparison

| Metric | Baseline (0.5) | Baseline (w+0.7) | EffNet (0.5) | EffNet (w+0.7) | DenseNet (0.5) | DenseNet (w+0.7) |
|---|---|---|---|---|---|---|
| **Accuracy** | 0.816 | **0.866** ↑ | 0.788 | 0.786 → | 0.884 | **0.896** ↑ |
| **Normal Recall** | 0.534 | **0.720** ↑ | 0.456 | 0.444 ↓ | 0.738 | **0.804** ↑ |
| **Pneumonia Recall** | 0.982 | 0.954 ↓ | 0.992 | 0.990 → | 0.972 | 0.954 ↓ |
| **Normal F1** | 0.678 | **0.800** ↑ | 0.614 | 0.586 ↓ | 0.828 | **0.854** ↑ |
| **Pneumonia F1** | 0.870 | **0.900** ↑ | 0.856 | 0.856 → | 0.914 | **0.920** ↑ |
| **Normal Recall Std** | 0.113 | 0.090 ↓ | 0.093 | **0.186** ↑↑ | 0.030 | 0.037 → |

---

## 6. Why EfficientNet Got Worse — The Critical Finding

EfficientNet is the only model that did not benefit from the intervention. This is the most important and most carefully examined finding of this section.

### What the data shows

```
Normal Recall:
EfficientNet without weighting: mean 0.456, std 0.093
EfficientNet with weighting:    mean 0.444, std 0.186

Normal Recall range:
Without weighting: 0.32 to 0.57  (spread of 0.25)
With weighting:    0.20 to 0.65  (spread of 0.45)
```

Two things happened simultaneously. The mean Normal Recall barely changed — 0.456 to 0.444, a negligible difference. But the standard deviation doubled from 0.093 to 0.186. The intervention did not lower performance on average. It made performance dramatically more unpredictable.

Run 1 gave Normal Recall of 0.20 — 80% of healthy patients incorrectly flagged. Run 2 gave 0.65 — acceptable performance. Run 5 gave 0.30. The same model architecture, same data, same intervention, producing results that differ by 45 percentage points across runs. Compare this to DenseNet which showed a range of only 10 percentage points (0.76 to 0.86) under the same intervention.

### The most defensible interpretation

Before this intervention, EfficientNet already showed the highest run-to-run variance of the transfer learning models. This was established in the unweighted experiments:

```
Normal Recall std without weighting:
DenseNet:     0.030  ← stable
EfficientNet: 0.093  ← already 3x more variable than DenseNet
```

EfficientNet's fine-tuning process was already sensitive to batch ordering before any intervention. Class weighting amplified this existing sensitivity. The amplified Normal gradient signal — 2.9x stronger per misclassified Normal case — interacted with an already unstable optimization landscape and made it more chaotic.

The most defensible interpretation is therefore: **class weighting destabilized a fine-tuning process that was already prone to instability.** DenseNet's fine-tuning process was already stable and absorbed the intervention without becoming unstable. EfficientNet's was not and did not.

### Why EfficientNet's fine-tuning was already unstable

This question connects back to findings established throughout the study. EfficientNet consistently showed higher run-to-run variance than DenseNet even without any intervention. The explanation offered throughout the study — that EfficientNet's sequential feature transformation does not preserve the fine-grained textural features needed for confident Normal detection — is a plausible architectural reason for this instability. When the model lacks stable, well-defined representations of Normal tissue, the optimization landscape during fine-tuning is more sensitive to small differences in which training examples appear in which order.

**Both EfficientNet and DenseNet use ImageNet pretrained weights in this study**. The difference is not the pretraining source — it is the architecture. DenseNet's dense connectivity means every layer retains direct access to all previous feature maps throughout the network. EfficientNet's compound scaling architecture progressively transforms and compresses features, with each layer only receiving input from the layer immediately before it.

Whether this specific architectural difference is the true cause of the instability difference cannot be proven from the experimental data alone. The data shows the outcome. The architectural reasoning is a plausible hypothesis consistent with the outcome and with everything observed throughout the study.

### My honest conclusion

EfficientNet did not benefit from class weighting and threshold adjustment. The variance doubled, making the model less reliable than it was before the intervention. The most defensible explanation is that class weighting amplified an existing instability in EfficientNet's fine-tuning process rather than guiding it toward better Normal representations. Whether this instability is fundamentally architectural, a consequence of hyperparameter choices, or some combination of both cannot be definitively determined from this study's data.

What can be said with confidence is that the intervention exposed a brittleness in EfficientNet that was not apparent from the unweighted results alone, and that this brittleness makes EfficientNet unsuitable for clinical deployment regardless of which configuration is used.

---

## 7. Why Baseline and DenseNet Improved

### Baseline improvement

The baseline benefited strongly from class weighting for a different reason than DenseNet. The baseline starts from random initialization and has no prior knowledge of either class. Class weighting effectively tells the training process to sample Normal cases more carefully — the larger loss contribution forces the optimizer to spend more of its capacity learning what Normal looks like rather than defaulting to Pneumonia. The result is a model that learns more balanced representations from scratch.

The threshold change (0.5 → 0.7) compounded this improvement by further clearing borderline cases toward Normal. The combination of forcing better Normal learning through weighting and then raising the confidence bar for Pneumonia predictions produced the largest improvement of any model — Normal Recall from 0.534 to 0.720.

The instability (std still 0.090) remains because random initialization is still the dominant source of variance, and weighting cannot eliminate that. But even the worst run (Normal Recall 0.59) is better than the original mean (0.534).

### DenseNet improvement

DenseNet improved because it had the representational capacity to take advantage of the rebalancing. Its dense connectivity already builds rich representations of Normal lung tissue — class weighting made the model pay even more attention to Normal during fine-tuning, pushing those already-good representations further. The result was a meaningful lift in Normal Recall from 0.738 to 0.804 with almost no increase in variance (std 0.030 → 0.037).

The Pneumonia Recall tradeoff (0.972 → 0.954) is the price of this improvement — the model is now less aggressively predicting Pneumonia for ambiguous cases. In a screening context this is an acceptable tradeoff: DenseNet still catches 95.4% of Pneumonia cases while correctly clearing 80.4% of healthy patients.

---

## 8. The Pneumonia Recall Tradeoff

All three models show a decrease in Pneumonia Recall with the intervention:

```
Baseline:  0.982 → 0.954  (−2.8 percentage points)
DenseNet:  0.972 → 0.954  (−1.8 percentage points)
```

This is expected and acceptable. The intervention deliberately pulled the model away from its Pneumonia-defaulting behavior, which by definition reduces Pneumonia Recall slightly. The question is whether the improvement in Normal Recall justifies the cost in Pneumonia Recall.

For the baseline: gaining 18.6 points of Normal Recall at the cost of 2.8 points of Pneumonia Recall is an excellent trade.

For DenseNet: gaining 6.6 points of Normal Recall at the cost of 1.8 points of Pneumonia Recall is a very good trade.

For EfficientNet: losing 0.8 points of Normal Recall while Pneumonia Recall stays essentially the same is no trade at all — the intervention did not work.

---

## 9. Final Model Rankings — With Intervention

With class weighting and threshold 0.7, the ranking changes slightly:

| Rank | Model | Mean Accuracy | Mean Normal Recall | Mean Pneumonia Recall |
|---|---|---|---|---|
| 1st | DenseNet-121 (weighted) | **0.896** | **0.804** | 0.954 |
| 2nd | Baseline CNN (weighted) | 0.866 | 0.720 | 0.954 |
| 3rd | EfficientNet-B0 (weighted) | 0.786 | 0.444 | **0.990** |

DenseNet remains the clear winner. The baseline makes significant gains and becomes meaningfully competitive. EfficientNet remains the weakest model and is further destabilized by the intervention.

The most surprising result is that the weighted baseline (Normal Recall 0.720) now outperforms the unweighted EfficientNet (Normal Recall 0.456) by a very wide margin. A simple CNN from scratch, when properly compensated for class imbalance, outperforms a pretrained 5-million parameter model that was not benefiting from the compensation. This is the clearest possible demonstration that architecture alone cannot overcome data problems.

---

## 10. Key Lessons

**Class weighting only helps models with sufficient representational capacity.** DenseNet benefited strongly. The baseline benefited strongly. EfficientNet did not benefit because it lacks good Normal feature representations. Amplifying the gradient signal for a feature you have not learned does not help — it creates instability.

**Threshold optimization is not neutral.** Raising the threshold systematically shifts the precision-recall tradeoff toward Normal. This is desirable when Normal Recall is the primary concern. Using a fixed arbitrary threshold like 0.5 leaves performance on the table.

**The two interventions interact.** Class weighting changes what the model learns. Threshold optimization changes how predictions are made from what the model learned. Both are needed for full effect. Class weighting without threshold adjustment would leave the learned representations underutilized at the default boundary. Threshold adjustment without class weighting would improve recall somewhat but not address the underlying learning bias.

**Data problems require data solutions — but data interventions require good models to take advantage of them.** DenseNet + class weighting + threshold optimization is the configuration that extracts the most value from all available tools: the right architecture, the right training signal, and the right decision boundary.